# Claude101 — Transcribir audio en el navegador / Transcribe audio in your browser

Sube un archivo de audio y obtén un transcript en `.txt` para usarlo con Claude. **No necesitas instalar nada en tu computadora.** Funciona en Windows, Mac, Linux, Chromebook — todo corre en Google Colab.

> Upload an audio file and get a `.txt` transcript to use with Claude. **You don't need to install anything on your machine.** Runs on Windows, Mac, Linux, Chromebook — everything runs in Google Colab.

## Instrucciones / Instructions

1. **Activa la GPU gratis / Enable free GPU:** menú **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)** · *Runtime → Change runtime type → GPU (T4)*
2. Corre las 4 celdas en orden (botón ▶ a la izquierda de cada celda) / Run the 4 cells in order (▶ on the left of each cell)
3. La primera celda tarda ~1 minuto / The first cell takes ~1 minute
4. El transcript se descarga solo cuando termina / The transcript downloads automatically when done

Modelo por defecto: **Whisper Large v3** (mejor calidad, detecta español automáticamente). Para audios largos en máquinas lentas, cambia a `medium` en la celda 3.

*Default model: **Whisper Large v3** (best quality, auto-detects Spanish). For long audios on slower runtimes, switch to `medium` in cell 3.*

## 1️⃣ Instalar Whisper / Install Whisper
Corre esta celda una sola vez. Tarda ~1 minuto. / Run this once. Takes ~1 minute.

In [ ]:
!pip install -q -U openai-whisper
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✓ Whisper instalado / installed. Usando / Using: {device.upper()}")
if device == "cpu":
    print("⚠ GPU no activada. Ve a Entorno de ejecución → Cambiar tipo de entorno → GPU (T4) y vuelve a correr esta celda.")
    print("⚠ GPU not enabled. Go to Runtime → Change runtime type → GPU (T4) and re-run this cell.")

## 2️⃣ Subir tu audio / Upload your audio
Haz clic en **Elegir archivos** y sube uno o varios audios (`.mp3`, `.m4a`, `.wav`, `.mp4`, etc).

*Click **Choose Files** and upload one or more audios (`.mp3`, `.m4a`, `.wav`, `.mp4`, etc).*

In [ ]:
from google.colab import files
uploaded = files.upload()
audio_files = list(uploaded.keys())
print(f"\n✓ {len(audio_files)} archivo(s) subido(s) / file(s) uploaded:")
for name in audio_files:
    size_mb = len(uploaded[name]) / 1024 / 1024
    print(f"   • {name}  ({size_mb:.1f} MB)")

## 3️⃣ Configurar y transcribir / Configure and transcribe

Cambia los valores en el panel derecho si necesitas. Por defecto: español, modelo Large v3.

*Change the values on the right panel if needed. Defaults: Spanish, Large v3 model.*

In [ ]:
#@title Opciones / Options { display-mode: "form" }
language = "Spanish" #@param ["Spanish", "English", "Portuguese", "French", "Italian", "German", "auto"]
model_size = "large-v3" #@param ["large-v3", "large-v3-turbo", "medium", "small", "base"]

import os, whisper, time

print(f"Cargando modelo / Loading model: {model_size} …")
t0 = time.time()
model = whisper.load_model(model_size, device=device)
print(f"✓ Modelo listo / Model ready in {time.time()-t0:.0f}s\n")

lang_arg = None if language == "auto" else language
transcripts = {}
for name in audio_files:
    print(f"▶ Transcribiendo / Transcribing: {name}")
    t0 = time.time()
    result = model.transcribe(
        name,
        language=lang_arg,
        condition_on_previous_text=False,
        verbose=False,
    )
    txt_name = os.path.splitext(name)[0] + ".txt"
    with open(txt_name, "w", encoding="utf-8") as f:
        f.write(result["text"].strip() + "\n")
    transcripts[name] = txt_name
    elapsed = time.time() - t0
    chars = len(result["text"])
    print(f"   ✓ {txt_name}  ({chars} caracteres / chars, {elapsed:.0f}s)")
    print(f"   Preview: {result['text'][:200].strip()}…\n")

## 4️⃣ Descargar transcripts / Download transcripts
Se descargan automáticamente como archivos `.txt`. Después, pégalos en Claude Desktop para extraer puntos clave, redactar el correo, etc.

*They download automatically as `.txt` files. Then paste them into Claude Desktop to extract key points, draft the follow-up email, etc.*

In [ ]:
from google.colab import files
for txt_name in transcripts.values():
    files.download(txt_name)
print("✓ Listo / Done.")

---

## Siguiente paso / Next step

Abre Claude Desktop y pega el contenido del `.txt`. Pídele:

> *"Extrae los puntos clave, las decisiones tomadas y los pendientes con responsable. Luego redacta un correo de seguimiento en español."*

*Open Claude Desktop and paste the `.txt` content. Ask it:*

> *"Extract key points, decisions made, and action items with owners. Then draft a follow-up email."*

---

## Problemas comunes / Troubleshooting

- **"GPU no activada" / "GPU not enabled":** Menú **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)**, guarda y vuelve a correr la celda 1.
- **El audio es muy largo / Audio too long:** Colab gratis tiene límite de ~12 GB RAM. Para audios > 1 hora, usa el modelo `medium` en lugar de `large-v3`.
- **El transcript tiene partes en otro idioma / Transcript has wrong language:** asegúrate de que en la celda 3 elegiste el idioma correcto (no dejes `auto`).